In [23]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_community.retrievers import BM25Retriever
from sentence_transformers import CrossEncoder

In [3]:
pdf_files = (Path.cwd().parent / "data").glob("*.pdf")

documents = []

for pdf_file in pdf_files:
    pdf_documents = PyPDFLoader(str(pdf_file)).load()
    documents.extend(pdf_documents)

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

for i , chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i

In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1465.09it/s]


In [7]:
vector_store = QdrantVectorStore.from_documents(
    chunks,
    embeddings,
    location=":memory:",
    collection_name="company_policy_data",
)

In [8]:
semantic_retriever = vector_store.as_retriever(
    search_kwargs={"k": 30}
)

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 30

In [9]:
def hybrid_retrieval(question, top_k=30):
    semantic_docs = semantic_retriever.invoke(question)
    bm25_docs = bm25_retriever.invoke(question)

    scores = {}
    docs = {}

    rrf_k = 60

    for rank,doc in enumerate(semantic_docs,start = 1):
        chunk_id = doc.metadata["chunk_id"]
        scores[chunk_id] = scores.get(chunk_id,0) + 1 / (rrf_k + rank)
        docs[chunk_id] = doc

    for rank,doc in enumerate(bm25_docs,start = 1):
        chunk_id = doc.metadata["chunk_id"]
        scores[chunk_id] = scores.get(chunk_id,0) + 1 / (rrf_k + rank)
        docs[chunk_id] = doc

    ranked_chunk_ids = sorted(
        scores,
        key = scores.get,
        reverse = True
    )

    return [docs[chunk_id] for chunk_id in ranked_chunk_ids[:top_k]]

In [10]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1359.24it/s]


In [11]:
def rerank_documents(question, docs, top_k=5):

    pairs = [
        (question, doc.page_content)
        for doc in docs
    ]

    scores = reranker.predict(pairs)

    scored_docs = list(zip(docs,scores))

    scored_docs.sort(key = lambda x:x[1],reverse=True)

    return [doc for doc,score in scored_docs[:top_k]]

In [12]:
def retrieve_documents(question, hybrid_top_k=30, final_top_k=5):

    hybrid_docs = hybrid_retrieval(question,top_k=hybrid_top_k)

    reranked_docs = rerank_documents(question,hybrid_docs,top_k=final_top_k)

    return reranked_docs

In [13]:
def format_docs(docs):
    return "\n\n".join(
        f"source: {doc.metadata['source']}\n"
        f"page: {doc.metadata['page'] + 1}\n"
        f"chunk_id: {doc.metadata['chunk_id']}\n"
        f"{doc.page_content}"
        for doc in docs
    )

In [14]:
prompt = ChatPromptTemplate.from_template("""
You are a question-answering assistant for Pexip Infinity documentation.

Answer the question using ONLY the provided context.

Rules:
- Do not use outside knowledge.
- - If the context does not contain enough information to answer the question, say:
  "I don't know based on the provided documentation."
- The answer may be supported by multiple chunks in the context. Combine information from all relevant chunks before deciding that you don't know.
- Give a clear and direct answer.
- Every factual claim must be supported by the provided context.
- At the end of your answer, list ALL supporting citations.
- Copy the source, page, and chunk_id exactly as they appear in the context.
- Do not invent or modify citation information.

Context:
{context}

Question:
{question}

Answer:
""")

In [15]:
llm = ChatOllama(
    model = "qwen2.5:7b"
)

In [16]:
rag_chain = (prompt | llm)

In [17]:
def ask_rag(question):
    docs = retrieve_documents(question,hybrid_top_k=30,final_top_k=5)
    context = format_docs(docs)

    response = rag_chain.invoke({"context":context,"question":question})

    return {"answer":response.content,
            "sources":[
                {
                "source": doc.metadata["source"],
                "page": doc.metadata["page"] + 1,
                "chunk_id": doc.metadata["chunk_id"]
                }
                for doc in docs 
            ]}

In [18]:
evaluation_dataset = [
    {
        "question": "How can an administrator access the Pexip Infinity Administrator interface?",
        "relevant_evidence": [
            {"chunk_id": 375, "page": 91},
            {"chunk_id": 376, "page": 91}
        ]
    },
    {
        "question": "What authentication methods are supported for administrator login to the Pexip Infinity Administrator interface?",
        "relevant_evidence": [
            {"chunk_id": 1741, "page": 384},
            {"chunk_id": 1742, "page": 384},
            {"chunk_id": 1743, "page": 384},
            {"chunk_id": 1744, "page": 384}
        ]
    },
    {
        "question": "What is a Management Node in Pexip Infinity?",
        "relevant_evidence": [
            {"chunk_id": 112, "page": 30},
            {"chunk_id": 114, "page": 30},
            {"chunk_id": 115, "page": 30}
        ]
    },
    {
        "question": "What is the role of a Conferencing Node in Pexip Infinity?",
        "relevant_evidence": [
            {"chunk_id": 684, "page": 153},
            {"chunk_id": 713, "page": 158},
            {"chunk_id": 731, "page": 161}
        ]
    },
    {
        "question": "How can a Conferencing Node be taken out of service?",
        "relevant_evidence": [
            {"chunk_id": 2801, "page": 483}
        ]
    },
    {
        "question": "What happens to existing calls when a Conferencing Node is put into maintenance mode?",
        "relevant_evidence": [
            {"chunk_id": 2138, "page": 483}
        ]
    },
    {
        "question": "What information can administrators view about a current conference?",
        "relevant_evidence": [
            {"chunk_id": 2425, "page": 555},
            {"chunk_id": 2427, "page": 555},
            {"chunk_id": 2428, "page": 555},
            {"chunk_id": 2429, "page": 555},
            {"chunk_id": 2430, "page": 555}
        ]
    },
    {
        "question": "What information can administrators view about completed or historical conferences?",
        "relevant_evidence": [
            {"chunk_id": 2478, "page": 568},
            {"chunk_id": 2479, "page": 568},
            {"chunk_id": 2480, "page": 568},
            {"chunk_id": 2481, "page": 568},
            {"chunk_id": 2482, "page": 568}
        ]
    },
    {
        "question": "What are the main functions administrators can perform through the Pexip Infinity Administrator interface?",
        "relevant_evidence": [
            {"chunk_id": 374, "page": 91}
        ]
    },
    {
        "question": "What happens to Pexip Infinity if the Management Node becomes unavailable?",
        "relevant_evidence": [
            {"chunk_id": 2252, "page": 505},
            {"chunk_id": 2253, "page": 505},
            {"chunk_id": 2254, "page": 505},
            {"chunk_id": 2255, "page": 505}
        ]
    }
]

In [19]:
def recall_at_k(retrieved_chunk_ids, relevant_evidence, k):

    relevant_chunk_ids = {
        evidence["chunk_id"]
        for evidence in relevant_evidence
    }

    retrieved_top_k = retrieved_chunk_ids[:k]

    relevant_retrieved = (
        set(retrieved_top_k)
        & relevant_chunk_ids
    )

    return (
        len(relevant_retrieved)
        / len(relevant_chunk_ids)
    )

In [20]:
def reciprocal_at_k(retrieved_chunk_ids, relevant_evidence):

    relevant_chunk_ids = {
        evidence["chunk_id"]
        for evidence in relevant_evidence
    }

    for rank, chunk_id in enumerate(
        retrieved_chunk_ids,
        start=1
    ):
        if chunk_id in relevant_chunk_ids:
            return 1 / rank

    return 0.0

In [21]:
def evaluate_retrieval(dataset, retriever_fn, k=10):

    recall_scores = []
    reciprocal_rank_scores = []

    for item in dataset:

        question = item["question"]
        relevant_evidence = item["relevant_evidence"]

        retrieved_docs = retriever_fn(question)

        retrieved_chunk_ids = [
            doc.metadata["chunk_id"]
            for doc in retrieved_docs[:k]
        ]

        recall = recall_at_k(
            retrieved_chunk_ids,
            relevant_evidence,
            k
        )

        rr = reciprocal_at_k(
            retrieved_chunk_ids,
            relevant_evidence
        )

        recall_scores.append(recall)
        reciprocal_rank_scores.append(rr)

    mean_recall = sum(recall_scores) / len(recall_scores)
    mrr = sum(reciprocal_rank_scores) / len(reciprocal_rank_scores)

    return mean_recall, mrr

In [22]:
if __name__ == "__main__":

    semantic_metrics = evaluate_retrieval(
        evaluation_dataset,
        lambda q: semantic_retriever.invoke(q),
        k=30
    )

    bm25_metrics = evaluate_retrieval(
        evaluation_dataset,
        lambda q: bm25_retriever.invoke(q),
        k=30
    )

    hybrid_metrics = evaluate_retrieval(
        evaluation_dataset,
        lambda q: hybrid_retrieval(q, top_k=10),
        k=10
    )

    reranked_metrics = evaluate_retrieval(
        evaluation_dataset,
        lambda q: retrieve_documents(
            q,
            hybrid_top_k=10,
            final_top_k=10
        ),
        k=10
    )

    print("Semantic:", semantic_metrics)
    print("BM25:", bm25_metrics)
    print("Hybrid:", hybrid_metrics)
    print("Reranked:", reranked_metrics)

Semantic: (0.5683333333333334, 0.35145021645021646)
BM25: (0.24833333333333335, 0.19974747474747473)
Hybrid: (0.34833333333333333, 0.3555555555555555)
Reranked: (0.34833333333333333, 0.3967857142857143)
